In [4]:
import geopandas as gpd

relavent_zones = [246, 50, 48, 158, 48, 186]
relavant_months = [8, 10, 11]
relavent_years = [2019, 2020, 2021, 2022, 2023,2024, 2025]

zones = gpd.read_file("taxi_zones/taxi_zones.shp")
zones_projected = zones.to_crs(epsg=2263)
zones_projected["centroid"] = zones_projected.geometry.centroid


# Now reproject just the centroid points to lat/lon for the weather API
zones_latlon = zones_projected.set_geometry("centroid").to_crs(epsg=4326)
zones_latlon["lat"] = zones_latlon.geometry.y
zones_latlon["lon"] = zones_latlon.geometry.x

zone_coords = zones_latlon[
    zones_latlon["LocationID"].isin(relavent_zones)
][["LocationID", "lat", "lon"]]

zone_coords.tail()

,LocationID,lat,lon
47,48,40.762253,-73.989845
49,50,40.766238,-73.995135
157,158,40.735035,-74.008984
185,186,40.748497,-73.992438
245,246,40.753309,-74.004016


In [5]:
# Weather grid resolution (Open-Meteo, ERA5, etc.) is typically ~9-11km. NYC's taxi zones are much smaller
# Fix: round each centroid to the weather source's grid resolution to avoid dupped calls 

zone_coords["grid_lat"] = zone_coords["lat"].round(1)  # ~11km grid
zone_coords["grid_lon"] = zone_coords["lon"].round(1)

unique_cells = zone_coords[["grid_lat", "grid_lon"]].drop_duplicates()
print(len(unique_cells))  # likely a small handful, not 263
unique_cells.head(18)

2


,grid_lat,grid_lon
47,40.8,-74.0
157,40.7,-74.0


In [7]:
import requests
import pandas as pd
import time
from pathlib import Path

cache_dir = Path('Data/weather_cache')
cache_dir.mkdir(parents=True, exist_ok=True)

relevant_periods = [
    *[(year, 10) for year in range(2019, 2026)],
    *[(year, 11) for year in range(2019, 2026)],
    (2024, 8),
    (2025, 8),
]

weather_frames = []

for _, cell in unique_cells.iterrows():
    lat, lon = cell["grid_lat"], cell["grid_lon"]

    for year, month in relevant_periods:
        cache_file = cache_dir / f"weather_{lat}_{lon}_{year}_{month:02d}.parquet"

        if cache_file.exists():
            df = pd.read_parquet(cache_file)
        else:
            start_date = pd.Timestamp(year=year, month=month, day=1)
            end_date = start_date + pd.offsets.MonthEnd(1)

            try:
                r = requests.get(
                    "https://archive-api.open-meteo.com/v1/archive",
                    params={
                        "latitude": lat,
                        "longitude": lon,
                        "start_date": start_date.strftime("%Y-%m-%d"),
                        "end_date": end_date.strftime("%Y-%m-%d"),
                        "hourly": "temperature_2m,precipitation,cloud_cover,wind_speed_10m",
                        "timezone": "America/New_York",
                    },
                    timeout=30,
                )
                r.raise_for_status()
            except requests.RequestException as e:
                print(f"Failed for {lat},{lon} {year}-{month:02d}: {e}")
                continue  # skip and keep going rather than crashing the whole run

            data = r.json().get("hourly")
            if data is None:
                print(f"No 'hourly' data for {lat},{lon} {year}-{month:02d}")
                continue

            df = pd.DataFrame(data)
            df["grid_lat"] = lat
            df["grid_lon"] = lon
            df.to_parquet(cache_file)  # cache to disk so re-runs are free
            time.sleep(0.5)  # be polite to the API

        weather_frames.append(df)

weather_df = pd.concat(weather_frames, ignore_index=True)
weather_df["time"] = pd.to_datetime(weather_df["time"])

In [8]:
weather_df.head()

,time,temperature_2m,precipitation,cloud_cover,wind_speed_10m,grid_lat,grid_lon
0,2019-10-01 00:00:00,17.5,0.1,98,9.0,40.8,-74.0
1,2019-10-01 01:00:00,17.4,0.0,100,11.2,40.8,-74.0
2,2019-10-01 02:00:00,17.2,0.0,10,10.8,40.8,-74.0
3,2019-10-01 03:00:00,16.6,0.0,100,6.2,40.8,-74.0
4,2019-10-01 04:00:00,17.3,0.0,100,8.0,40.8,-74.0


In [9]:
zone_weather_map = zone_coords.merge(weather_df, on=["grid_lat", "grid_lon"], how="left")
zone_weather_map.sample(10)

# columns: LocationID, time, temperature_2m, precipitation, cloud_cover, wind_speed_10m


,LocationID,lat,lon,grid_lat,grid_lon,time,temperature_2m,precipitation,cloud_cover,wind_speed_10m
1935,48,40.762253,-73.989845,40.8,-74.0,2021-10-19 15:00:00,19.1,0.0,0,24.6
17,48,40.762253,-73.989845,40.8,-74.0,2019-10-01 17:00:00,25.9,0.0,43,14.4
55451,246,40.753309,-74.004016,40.8,-74.0,2023-11-18 11:00:00,11.1,0.1,100,23.0
27742,158,40.735035,-74.008984,40.7,-74.0,2024-10-23 22:00:00,17.6,0.0,1,11.1
26946,158,40.735035,-74.008984,40.7,-74.0,2023-10-21 18:00:00,15.9,0.0,7,20.6
43667,186,40.748497,-73.992438,40.7,-74.0,2023-11-16 11:00:00,11.6,0.0,0,5.0
42735,186,40.748497,-73.992438,40.7,-74.0,2022-11-07 15:00:00,23.6,0.0,0,19.3
44131,186,40.748497,-73.992438,40.7,-74.0,2024-11-05 19:00:00,19.6,0.0,0,13.4
18332,50,40.766238,-73.995135,40.8,-74.0,2020-11-28 20:00:00,7.1,0.0,0,12.0
40645,186,40.748497,-73.992438,40.7,-74.0,2019-11-10 13:00:00,8.6,0.0,56,14.5


In [11]:
# use time and locationID as the join keys to merge with the trips data in Spark
zone_weather_map.to_parquet("Data/zone_weather_map.parquet", index=False)